# 03 — Testing the updated algorithm on the SRSR dataset

This notebook performs an end-to-end evaluation of the current deduplication algorithm on the **SRSR (Systematic Review of Systematic Reviews)** dataset — the same labelled corpus that was used to derive the logistic regression weights in `01_building_dedupe_feature_importance_model.ipynb`.

## Purpose and scope

Since first fitting the weights, several improvements have been layered on top of the base logistic regression scorer:

- **Early-stopping rules** — pairs are immediately scored 0 when there is strong evidence against a duplicate (e.g. DOI and pages both disagree, title similarity below a veto threshold, year gap > 1 combined with conflicting abstract content).
- **DOI normalisation** — case folding, URL-prefix stripping, and a fuzzy punctuation-stripped fallback to handle common database artefacts noticed in `02_explore_doi_discrepancies.ipynb` (missing dots, underscore/hyphen substitution).
- **Page normalisation** — canonicalisation of shorthand ranges and unicode dashes.

Because this is the **same dataset used for model development**, results here represent an optimistic upper bound on real-world performance (in-sample evaluation). The goal is to confirm the algorithm is not degraded by the rule additions before moving to held-out unseen datasets.

## Evaluation strategy

We use two complementary evaluation levels:

1. **Pair-level** — classic precision/recall/F1 on the set of blocked candidate pairs; computed for a range of score thresholds.
2. **Record-level (ASySD-style)** — following Hair et al. (2023), pairs are clustered into connected components and each cluster retains exactly one record. The resulting kept/removed decisions are compared against gold-standard duplicate groups to build a record-level confusion matrix.

The record-level view is more directly interpretable: a false positive means a real unique paper was accidentally discarded, and a false negative means a real duplicate survived deduplication.

## Setup

Imports cover three areas:

- **Standard data science stack** (`pandas`, `numpy`, `sklearn`) for data handling and computing evaluation metrics.
- **App modules** — `Deduper` is the main deduplication class; `BLOCK_RULES` defines which field combinations are used for candidate-pair blocking; `ExtendedPaper` is the Pydantic model that adds gold-standard fields (`recordid`, `duplicateid`) on top of the base `Paper` schema.
- **Path configuration** — the notebook resolves the repository root dynamically so it can be run from either the `notebooks/` folder or the repo root.

In [1]:
from __future__ import annotations

import re
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from pydantic import ValidationError
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from tqdm import tqdm

# Ensure imports like "from app..." work when notebook is opened from notebooks/
repo_root = Path.cwd()
if not (repo_root / "app").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from app.dedupe import Deduper
from app.determine_weights import BLOCK_RULES, BLOCK_RULES_OLD, ExtendedPaper
from app.normalisers import normalize_doi, normalize_pages, strip_doi_punctuation

DATA_PATH = repo_root / "notebooks/data/srsr_data.csv"


## Data loading and record cache

The SRSR dataset (`notebooks/data/srsr_data.csv`) contains **53,001 records** from a preclinical systematic review search, with expert-labelled duplicate groups. Each duplicate group shares a common `duplicateid`; records with a unique or missing `duplicateid` are true uniques.

Two helper functions are defined here:

- **`load_data`** — reads the CSV, normalises column names (strips BOM characters, lowercases), and renames `author`→`authors` and `number`→`issue` to match the `Paper` schema.
- **`build_record_cache`** — parses every row into an `ExtendedPaper` Pydantic object and stores them in a `dict` keyed by `recordid`. Pre-building this dictionary means each record is parsed only once, making bulk pair scoring O(1) per lookup rather than O(n) per pair.

In [2]:
def norm_value(field: str, val) -> str | None:
    if pd.isna(val) or val is None or val == "":
        return None

    text = str(val).strip()
    if text == "":
        return None

    # Field-aware normalization before blocking
    if field == "doi":
        doi = normalize_doi(text)
        return strip_doi_punctuation(doi) if doi else None

    if field == "pages":
        pages = normalize_pages(text)
        return re.sub(r"\s+", "", pages).lower() if pages else None

    if field in {"year", "volume", "issue"}:
        cleaned = re.sub(r"\s+", "", text)
        try:
            return str(int(float(cleaned)))
        except ValueError:
            return cleaned.lower()

    return re.sub(r"\s+", " ", text.lower())


def _sanitize_row_for_pydantic(record: dict) -> dict:
    sanitized = {}
    for key, value in record.items():
        if pd.isna(value):
            sanitized[key] = None
            continue

        if isinstance(value, str):
            stripped = value.strip()
            value = stripped if stripped else None

        if key == "doi" and value is not None:
            normalized = normalize_doi(str(value))
            sanitized[key] = normalized if normalized else None
        else:
            sanitized[key] = value
    return sanitized


def build_record_cache(
    df: pd.DataFrame,
    id_column: str = "recordid",
    include_ids: set[int] | None = None,
 ) -> dict[int, ExtendedPaper]:
    cache: dict[int, ExtendedPaper] = {}
    validation_errors = 0

    for record in df.to_dict(orient="records"):
        record_id = record.get(id_column)
        if record_id is None or pd.isna(record_id):
            continue

        parsed_id = int(record_id)
        if include_ids is not None and parsed_id not in include_ids:
            continue

        cleaned_record = _sanitize_row_for_pydantic(record)
        try:
            cache[parsed_id] = ExtendedPaper(**cleaned_record)
        except ValidationError:
            validation_errors += 1

    if validation_errors:
        print(f"Skipped {validation_errors} invalid records while building the cache")

    return cache


def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="latin-1")
    df.columns = [c.replace("ï..", "").strip().lower() for c in df.columns]
    return df.rename(columns={"author": "authors", "number": "issue"})


df = load_data(DATA_PATH)
print(df.shape)
df.head()


(53001, 23)


,unnamed: 0,,authors,year,journal,doi,title,pages,volume,issue,...,label,endnote,bond,asysd,gold,duplicateid,nbond,nendnote,nasysd,ngold
0,1,20363,Slavin S. A.Lin S. J.,2012.0,Plast Reconstr Surg,10.1097/PRS.0b013e31825f23ca,THE USE OF ACELLULAR DERMAL MATRICES IN REVISI...,70s-85s,130,5 Suppl 2,...,NaN,KEEP,REMOVE,KEEP,KEEP,15153,0,1,1,1
1,2,23210,Wang K. K.Donahue T. R.Haber G. B.DeWitt J. M....,2017.0,Gastroenterology,NaN,THE USE OF PHOTODYNAMIC THERAPY IN PANCREATIC ...,S498,152 (5 Supplement 1),NaN,...,NaN,KEEP,REMOVE,KEEP,KEEP,29964,0,1,1,1
2,3,23459,Watson L. I.Armon M. P.,2004.0,Cochrane Database Syst Rev,10.1002/14651858.CD002783.pub2,THROMBOLYSIS FOR ACUTE DEEP VEIN THROMBOSIS,Cd002783,NaN,4,...,NaN,KEEP,REMOVE,KEEP,KEEP,22076,0,1,1,1
3,4,7212,Gagyor I.Madhok V. B.Daly F.Somasundara D.Sull...,2015.0,Cochrane Database Syst Rev,10.1002/14651858.CD001869.pub5,WITHDRAWN. ANTIVIRAL TREATMENT FOR BELL'S PALS...,Cd001869,NaN,5,...,REMOVED,KEEP,REMOVE,KEEP,REMOVE,10934,0,1,1,0
4,5,46468,Wang E. E.Tang N. K.,2007.0,Cochrane database of systematic reviews (Online),NaN,WITHDRAWN: IMMUNOGLOBULIN FOR PREVENTING RESPI...,CD001725,NaN,3,...,REMOVED,REMOVE,REMOVE,REMOVE,REMOVE,20457,0,1,1,1


## Blocking: generating candidate pairs

Comparing all $\binom{53001}{2} \approx 1.4$ billion record pairs is computationally infeasible. **Blocking** restricts comparisons to pairs that agree on at least one field combination defined in `BLOCK_RULES`. The current rules are:

| Rule fields | Rationale |
|---|---|
| `title` | Records sharing a normalised title token are very likely duplicates |
| `abstract` | Shared abstract text is a strong duplicate signal |
| `doi` | Exact (normalised) DOI match |
| `year` + `journal` | Same year and journal |
| `year` + `pages` | Same year and page range |
| `year` + `volume` | Same year and volume |
| `pages` + `volume` | Same pages and volume |
| `pages` + `issue` | Same pages and issue |
| `year` + `issue` | Same year and issue |

Before grouping, values are normalised with field-aware logic:

- DOI: canonical DOI extraction plus punctuation-stripped suffix fallback
- Pages: canonical page-range normalisation
- Year/Volume/Issue: numeric canonicalisation (e.g. `01` -> `1`)
- Other fields: lowercase and collapsed whitespace

Pairs caught by multiple rules are deduplicated so each `(id_a, id_b)` appears exactly once in the output, but all matched rules are recorded in `block_rules` for later inspection.

The summary table shows, per rule, how many total pairs were generated and how many of those are genuine duplicates (`is_dupe=1`). The `unique_dupes` column counts duplicates caught **exclusively** by that rule, which indicates how many true duplicates would be missed if it were removed.

In [3]:
def build_blocked_pairs(df: pd.DataFrame) -> pd.DataFrame:
    dup_lookup = df.set_index("recordid")["duplicateid"].to_dict()
    seen = set()
    rows = []
    pair_rules: dict[tuple, set[str]] = {}

    for rule in BLOCK_RULES_OLD:
        missing = [field for field in rule if field not in df.columns]
        if missing:
            continue

        rule_label = ",".join(rule)  # e.g. "year,journal"
        subset = df[["recordid", *rule]].copy()
        norm_cols = []
        for field in rule:
            norm_col = f"{field}_norm"
            subset[norm_col] = subset[field].apply(
                lambda v, field_name=field: norm_value(field_name, v)
            )
            norm_cols.append(norm_col)

        subset = subset.dropna(subset=norm_cols)
        subset["block_key"] = subset[norm_cols].apply(tuple, axis=1)

        for _, group in subset.groupby("block_key"):
            ids = group["recordid"].tolist()
            if len(ids) < 2:
                continue

            for a, b in combinations(ids, 2):
                id_a, id_b = (a, b) if a < b else (b, a)
                key = (id_a, id_b)
                pair_rules.setdefault(key, set()).add(rule_label)
                if key in seen:
                    continue

                dup_a = dup_lookup.get(id_a)
                dup_b = dup_lookup.get(id_b)
                is_dupe = int(pd.notna(dup_a) and pd.notna(dup_b) and dup_a == dup_b)

                rows.append((id_a, id_b, is_dupe))
                seen.add(key)

    all_pairs_df = pd.DataFrame(rows, columns=["id_a", "id_b", "is_dupe"])
    all_pairs_df["block_rules"] = all_pairs_df.apply(
        lambda r: " | ".join(sorted(pair_rules.get((r["id_a"], r["id_b"]), set()))), axis=1
    )
    return all_pairs_df


all_pairs_df = build_blocked_pairs(df)

# Overall totals
n_dupes = all_pairs_df["is_dupe"].sum()
n_non_dupes = (all_pairs_df["is_dupe"] == 0).sum()
print(f"Total pairs: {len(all_pairs_df):,}  |  dupes: {n_dupes:,}  |  non-dupes: {n_non_dupes:,}")

# Per-rule counts (each pair counted once per rule it matched)
exploded = all_pairs_df.copy()
exploded["rule"] = exploded["block_rules"].str.split(" | ", regex=False)
exploded = exploded.explode("rule")

rule_summary = (
    exploded.groupby("rule")["is_dupe"]
    .agg(dupes="sum", non_dupes=lambda x: (x == 0).sum())
    .assign(total=lambda d: d["dupes"] + d["non_dupes"])
)

# Unique pairs = caught by exactly one rule (no " | " in block_rules)
unique_only = exploded[~exploded["block_rules"].str.contains(" | ", regex=False)]
unique_summary = unique_only.groupby("rule")["is_dupe"].agg(
    unique_dupes="sum",
    unique_non_dupes=lambda x: (x == 0).sum(),
)

rule_summary = (
    rule_summary.join(unique_summary)
    .fillna(0)
    .astype({"unique_dupes": int, "unique_non_dupes": int})
    .sort_values("total", ascending=False)
)

# Leave-one-rule-out impact: what would be lost if this rule is removed?
rule_removal_impact = rule_summary.copy()
rule_removal_impact["pairs_lost_if_removed"] = (
    rule_removal_impact["unique_dupes"] + rule_removal_impact["unique_non_dupes"]
)
rule_removal_impact["dupes_lost_if_removed"] = rule_removal_impact["unique_dupes"]
rule_removal_impact["remaining_pairs"] = len(all_pairs_df) - rule_removal_impact["pairs_lost_if_removed"]
rule_removal_impact["remaining_dupes"] = int(n_dupes) - rule_removal_impact["dupes_lost_if_removed"]
rule_removal_impact["dupe_recall_after_removal"] = (
    rule_removal_impact["remaining_dupes"] / max(int(n_dupes), 1)
)
rule_removal_impact["pair_set_reduction_pct"] = (
    100 * rule_removal_impact["pairs_lost_if_removed"] / max(len(all_pairs_df), 1)
).round(2)
rule_removal_impact["dupe_loss_pct"] = (
    100 * rule_removal_impact["dupes_lost_if_removed"] / max(int(n_dupes), 1)
).round(2)

display(rule_summary)
display(
    rule_removal_impact[
        [
            "dupes",
            "non_dupes",
            "pairs_lost_if_removed",
            "dupes_lost_if_removed",
            "pair_set_reduction_pct",
            "dupe_loss_pct",
            "dupe_recall_after_removal",
        ]
    ].sort_values(["dupes_lost_if_removed", "pairs_lost_if_removed"], ascending=[True, False])
)


Total pairs: 5,214,643  |  dupes: 22,696  |  non-dupes: 5,191,947


,dupes,non_dupes,total,unique_dupes,unique_non_dupes
rule,,,,,
"year,issue",15755,4242549,4258304,2,4179295
"year,volume",19036,926981,946017,2,772072
"year,journal",16536,169679,186215,9,62436
"pages,issue",14432,17306,31738,0,15944
"year,pages",17277,4785,22062,6,3571
title,20939,633,21572,31,566
doi,20948,493,21441,31,2
"pages,volume",16814,812,17626,1,468
abstract,943,36,979,1,20


,dupes,non_dupes,pairs_lost_if_removed,dupes_lost_if_removed,pair_set_reduction_pct,dupe_loss_pct,dupe_recall_after_removal
rule,,,,,,,
"pages,issue",14432,17306,15944,0,0.31,0.00,1.000000
"pages,volume",16814,812,469,1,0.01,0.00,0.999956
abstract,943,36,21,1,0.00,0.00,0.999956
"year,issue",15755,4242549,4179297,2,80.15,0.01,0.999912
"year,volume",19036,926981,772074,2,14.81,0.01,0.999912
"year,pages",17277,4785,3577,6,0.07,0.03,0.999736
"year,journal",16536,169679,62445,9,1.20,0.04,0.999603
title,20939,633,597,31,0.01,0.14,0.998634
doi,20948,493,33,31,0.00,0.14,0.998634


Use the second output table (`rule_removal_impact`) to decide whether a rule is safe to remove.

Key columns to focus on:

- **`dupes_lost_if_removed`**: true duplicates that would no longer be blocked at all
- **`dupe_loss_pct`**: percentage duplicate loss from removing that rule
- **`pair_set_reduction_pct`**: how much total pair volume (and scoring cost) you save
- **`dupe_recall_after_removal`**: duplicate coverage retained after removal

A practical rule-removal candidate usually has **high `pair_set_reduction_pct`** and **near-zero `dupes_lost_if_removed`**.

For this reason, we have removed pages+issue and year+issue as blocking criteria 

In [4]:
def build_blocked_pairs(df: pd.DataFrame) -> pd.DataFrame:
    dup_lookup = df.set_index("recordid")["duplicateid"].to_dict()
    seen = set()
    rows = []
    pair_rules: dict[tuple, set[str]] = {}

    for rule in BLOCK_RULES:
        missing = [field for field in rule if field not in df.columns]
        if missing:
            continue

        rule_label = ",".join(rule)
        subset = df[["recordid", *rule]].copy()
        norm_cols = []
        for field in rule:
            norm_col = f"{field}_norm"
            subset[norm_col] = subset[field].apply(
                lambda v, field_name=field: norm_value(field_name, v)
            )
            norm_cols.append(norm_col)

        subset = subset.dropna(subset=norm_cols)
        subset["block_key"] = subset[norm_cols].apply(tuple, axis=1)

        for _, group in subset.groupby("block_key"):
            ids = group["recordid"].tolist()
            if len(ids) < 2:
                continue

            for a, b in combinations(ids, 2):
                id_a, id_b = (a, b) if a < b else (b, a)
                key = (id_a, id_b)
                pair_rules.setdefault(key, set()).add(rule_label)
                if key in seen:
                    continue

                dup_a = dup_lookup.get(id_a)
                dup_b = dup_lookup.get(id_b)
                is_dupe = int(pd.notna(dup_a) and pd.notna(dup_b) and dup_a == dup_b)

                rows.append((id_a, id_b, is_dupe))
                seen.add(key)

    all_pairs_df = pd.DataFrame(rows, columns=["id_a", "id_b", "is_dupe"])
    all_pairs_df["block_rules"] = all_pairs_df.apply(
        lambda r: " | ".join(sorted(pair_rules.get((r["id_a"], r["id_b"]), set()))), axis=1
    )
    return all_pairs_df


all_pairs_df = build_blocked_pairs(df)

# Overall totals
n_dupes = all_pairs_df["is_dupe"].sum()
n_non_dupes = (all_pairs_df["is_dupe"] == 0).sum()
print(f"Total pairs: {len(all_pairs_df):,}  |  dupes: {n_dupes:,}  |  non-dupes: {n_non_dupes:,}")

Total pairs: 1,019,402  |  dupes: 22,694  |  non-dupes: 996,708


## Pair selection

`MAX_PAIRS` controls whether to work with all blocked pairs or a random subsample. Set it to `None` (default) to score every pair; set it to a fixed integer to cap the workload during development or quick iteration. When a subsample is used, the gold-standard `is_dupe` labels are preserved, so all downstream metrics remain valid — they just reflect a subset of the full candidate space.

In [5]:
MAX_PAIRS = None  # set to None to use all blocked pairs

if MAX_PAIRS is not None and len(all_pairs_df) > MAX_PAIRS:
    pairs_df = all_pairs_df.sample(n=MAX_PAIRS, random_state=1234).reset_index(drop=True)
else:
    pairs_df = all_pairs_df.copy()

print(pairs_df.shape)
pairs_df.head()


(1019402, 4)


,id_a,id_b,is_dupe,block_rules
0,20873,36671,1,"doi | pages,volume | title | year,pages | year..."
1,20873,55755,1,"doi | pages,volume | title | year,pages | year..."
2,36671,55755,1,"doi | pages,volume | title | year,journal | ye..."
3,22749,37428,1,"abstract | pages,volume | title | year,journal..."
4,21949,36769,1,"title | year,pages"


## Scoring pairs

Each candidate pair is scored using `Deduper.dedupe_weighted()`, which applies the logistic regression model derived in `01_building_dedupe_feature_importance_model.ipynb`:

$$P(\text{duplicate}) = \sigma\!\left(\sum_{f} w_f \cdot s_f + b\right)$$

where $s_f \in [0, 1]$ is the field similarity score, $w_f$ is the learned weight for that field, and $b$ is the intercept. $\sigma$ is the sigmoid function.

**`issue` and `abstract` are excluded** from the weighted sum here — both have negative or near-zero weights in the fitted model and were found to add noise rather than signal when used in the scoring step.

The early-stopping rules are applied before scoring. If a pair triggers an early stop (e.g. DOI and pages both disagree), it is immediately assigned a score of 0.0 without computing any field similarities, which substantially reduces runtime on large datasets.

In [6]:
import importlib
import app.dedupe as _dedupe_mod
importlib.reload(_dedupe_mod)

from loguru import logger
from app.dedupe import Deduper, WEIGHTS, INTERCEPT

# Weights excluding 'issue' and 'abstract'
WEIGHTS_FILTERED = {k: v for k, v in WEIGHTS.items() if k not in ("issue", "abstract")}

SCORE_FIELDS = list(WEIGHTS_FILTERED.keys())  # fields whose individual scores will be exported

def score_pairs_weighted(df: pd.DataFrame, pairs_df: pd.DataFrame) -> pd.DataFrame:
    """Score pairs using Deduper's weighted dedupe logic, excluding 'issue' and 'abstract'."""
    record_cache = build_record_cache(df, id_column="recordid")

    any_paper = next(iter(record_cache.values()))
    deduper = Deduper(reference=any_paper, candidates=[any_paper])

    results = []
    logger.disable("app.dedupe")
    try:
        for row in tqdm(pairs_df.itertuples(index=False), total=len(pairs_df)):
            rec_a = record_cache.get(int(row.id_a))
            rec_b = record_cache.get(int(row.id_b))
            if rec_a is None or rec_b is None:
                continue
            prob, field_scores, early_stop = deduper.score_pair(
                rec_a, rec_b, weights=WEIGHTS_FILTERED, intercept=INTERCEPT
            )
            result = {
                "id_a": row.id_a,
                "id_b": row.id_b,
                "is_dupe": row.is_dupe,
                "prob": prob,
                "early_stop": early_stop,
            }
            for field in SCORE_FIELDS:
                result[f"score_{field}"] = field_scores.get(field)
            results.append(result)
    finally:
        logger.enable("app.dedupe")

    return pd.DataFrame(results)

scored_df = score_pairs_weighted(df, pairs_df)
scored_df.head()


Skipped 21 invalid records while building the cache


100%|██████████| 1019402/1019402 [00:36<00:00, 27707.08it/s]


,id_a,id_b,is_dupe,prob,early_stop,score_doi,score_title,score_authors,score_year,score_journal,score_pages,score_volume
0,20873,36671,1,0.993419,None,1.0,1.0,1.0,1.0,0.850000,1.0,1.0
1,20873,55755,1,0.993419,None,1.0,1.0,1.0,1.0,0.850000,1.0,1.0
2,36671,55755,1,0.994675,None,1.0,1.0,1.0,1.0,1.000000,1.0,1.0
3,22749,37428,1,0.950263,None,NaN,1.0,1.0,1.0,1.000000,1.0,1.0
4,21949,36769,1,0.915054,None,NaN,1.0,1.0,1.0,0.864061,1.0,NaN


## Pair-level evaluation across thresholds

For each score threshold in `THRESHOLDS`, a pair is classified as a predicted duplicate if `prob >= threshold`. We compute:

| Metric | Definition |
|---|---|
| **Sensitivity** (recall) | $\frac{TP}{TP + FN}$ — fraction of true duplicate pairs detected |
| **Specificity** | $\frac{TN}{TN + FP}$ — fraction of non-duplicate pairs correctly rejected |
| **Precision** | $\frac{TP}{TP + FP}$ — fraction of predicted duplicates that are genuine |
| **ROC-AUC** | Threshold-agnostic ranking quality |
| **Average precision** | Area under the precision-recall curve |

False positive and false negative pairs are exported at each selected threshold to support qualitative inspection:

- `notebooks/results/srsr_dataset_run/false_positives_{threshold}.csv`
- `notebooks/results/srsr_dataset_run/false_negatives_{threshold}.csv`

> **Note on in-sample bias:** because the weights were derived from this same dataset, the pair-level metrics will be optimistically inflated compared to a held-out test set. The record-level evaluation below is less sensitive to this because it measures the downstream outcome (which records survive deduplication) rather than raw pair classification accuracy.

In [7]:
THRESHOLDS = [0.7, 0.75, 0.8, 0.85, 0.90]

def metrics_for_threshold(scored_df: pd.DataFrame, threshold: float) -> dict:
    y_true = scored_df["is_dupe"].astype(int)
    y_prob = scored_df["prob"].astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0

    return {
        "threshold": threshold,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "average_precision": float(average_precision_score(y_true, y_prob)),
    }

import os

def _pair_side_by_side_export(df, pair_df, fields, out_path):
    # Metadata columns to carry through from scored_df
    meta_cols = [c for c in ["prob", "early_stop"] + [f"score_{f}" for f in SCORE_FIELDS] if c in pair_df.columns]

    left = df[["recordid"] + fields].copy()
    left.columns = ["id_a"] + [f"{f}_a" for f in fields]
    right = df[["recordid"] + fields].copy()
    right.columns = ["id_b"] + [f"{f}_b" for f in fields]

    export = pair_df[["id_a", "id_b"] + meta_cols].merge(left, on="id_a", how="left").merge(right, on="id_b", how="left")

    # Side-by-side field pairs first, then scores
    side_by_side = ["id_a", "id_b"] + meta_cols + [
        col
        for f in fields
        for col in (f"{f}_a", f"{f}_b")
        if col in export.columns
    ]
    export_subset = export[[c for c in side_by_side if c in export.columns]]
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    export_subset.to_csv(out_path, index=False)
    return out_path

def export_false_positives_side_by_side(df, scored_df, fields, threshold, out_path):
    y_true = scored_df["is_dupe"].astype(int)
    y_pred = (scored_df["prob"].astype(float) >= threshold).astype(int)
    fp_df = scored_df[(y_true == 0) & (y_pred == 1)].copy()
    return _pair_side_by_side_export(df, fp_df, fields, out_path)

def export_false_negatives_side_by_side(df, scored_df, fields, threshold, out_path):
    y_true = scored_df["is_dupe"].astype(int)
    y_pred = (scored_df["prob"].astype(float) >= threshold).astype(int)
    fn_df = scored_df[(y_true == 1) & (y_pred == 0)].copy()
    return _pair_side_by_side_export(df, fn_df, fields, out_path)

metrics = []
for threshold in THRESHOLDS:
    row = metrics_for_threshold(scored_df, threshold)
    metrics.append(row)
    export_false_positives_side_by_side(df, scored_df, SCORE_FIELDS, threshold, f"results/srsr_dataset_run/false_positives_{threshold}.csv")
    export_false_negatives_side_by_side(df, scored_df, SCORE_FIELDS, threshold, f"results/srsr_dataset_run/false_negatives_{threshold}.csv")

metrics_df = pd.DataFrame(metrics)
metrics_df


,threshold,tp,fp,tn,fn,sensitivity,specificity,precision,roc_auc,average_precision
0,0.70,22604,106,996044,85,0.996254,0.999894,0.995332,0.999137,0.998096
1,0.75,22566,72,996078,123,0.994579,0.999928,0.996820,0.999137,0.998096
2,0.80,22510,47,996103,179,0.992111,0.999953,0.997916,0.999137,0.998096
3,0.85,22331,20,996130,358,0.984221,0.999980,0.999105,0.999137,0.998096
4,0.90,22087,13,996137,602,0.973467,0.999987,0.999412,0.999137,0.998096


## Selecting the operating threshold

For deduplication in systematic review pipelines, the primary concern is **not discarding genuine unique records** (i.e. minimising false positives at the record level). We therefore search for the threshold that maximises **specificity** subject to a minimum sensitivity constraint of 98% — meaning at most 2% of true duplicates are allowed to survive.

The `find_best_threshold` function scans the metrics table and returns the threshold that best satisfies this trade-off. If no threshold meets the minimum sensitivity requirement, it reports this explicitly so the threshold can be reconsidered.

In [8]:
def find_best_threshold(metrics_df, min_sensitivity=0.99):
    """
    Find the threshold with the highest specificity where sensitivity >= min_sensitivity.
    Returns a dict with the best row, or None if no threshold meets the criteria.
    """
    filtered = metrics_df[metrics_df['sensitivity'] >= min_sensitivity]
    if filtered.empty:
        return None
    best_row = filtered.loc[filtered['specificity'].idxmax()]
    return best_row

# Example usage after metrics_df is created:
best = find_best_threshold(metrics_df, min_sensitivity=0.99)
if best is not None:
    print(f"Best threshold with sensitivity >= 0.99: {best['threshold']:.3f}")
    print(best)
else:
    print("No threshold found with sensitivity >= 0.99")


Best threshold with sensitivity >= 0.99: 0.800
threshold                 0.800000
tp                    22510.000000
fp                       47.000000
tn                   996103.000000
fn                      179.000000
sensitivity               0.992111
specificity               0.999953
precision                 0.997916
roc_auc                   0.999137
average_precision         0.998096
Name: 2, dtype: float64


## Record-level evaluation (ASySD-style)

The pair-level metrics above tell us how well the model ranks pairs, but they do not directly answer the question a researcher cares about: *which records will I end up with after deduplication?*

We adopt the evaluation protocol from Hair et al. (2023) used in the [ASySD R package](https://github.com/camaradesuk/ASySD):

1. **Cluster** — scored pairs above the threshold are treated as edges in a graph; connected components become predicted duplicate groups.
2. **Retain one per cluster** — within each predicted group, the record with the lowest `recordid` is kept and all others are marked as removed. (The choice of which record to retain does not affect the confusion matrix.)
3. **Gold standard** — records sharing a `duplicateid` form a true duplicate group; one record per group should be kept and the rest should be removed. Records with a unique or missing `duplicateid` are true uniques and should always be kept.
4. **Confusion matrix** at the record level:

| | **Predicted: removed** | **Predicted: kept** |
|---|---|---|
| **True: duplicate** | TP — correctly removed | FN — missed duplicate |
| **True: unique** | FP — wrongly removed | TN — correctly kept |

The key metrics are:
- **Sensitivity** $= \frac{TP}{TP+FN}$ — proportion of true duplicates that were successfully removed.
- **Specificity** $= \frac{TN}{TN+FP}$ — proportion of true unique records that were correctly retained. A single false positive here means a real paper was lost from the search results.
- **Precision** $= \frac{TP}{TP+FP}$ — of all records removed, what fraction were genuine duplicates.

In [9]:
import networkx as nx

THRESHOLD = 0.8  # adjust as needed

# --- Step 1: Cluster positive pairs into connected components ---
def cluster_records(scored_df: pd.DataFrame, threshold: float, all_record_ids: set) -> pd.Series:
    """Return a Series mapping recordid -> predicted_group (integer)."""
    pos_pairs = scored_df[scored_df['prob'] >= threshold][['id_a', 'id_b']].to_numpy()
    G = nx.Graph()
    G.add_nodes_from(all_record_ids)
    G.add_edges_from(pos_pairs)
    record_to_group = {}
    for group_id, component in enumerate(nx.connected_components(G)):
        for rid in component:
            record_to_group[rid] = group_id
    return pd.Series(record_to_group, name='predicted_group')

all_record_ids = set(df['recordid'])
record_to_group = cluster_records(scored_df, THRESHOLD, all_record_ids)
df['predicted_group'] = df['recordid'].map(record_to_group)

# For each predicted cluster, retain the record with the lowest recordid (ASySD convention)
df['pred_keep'] = df.groupby('predicted_group')['recordid'].transform('min') == df['recordid']

print(f"Total records: {len(df)}")
print(f"Predicted clusters: {df['predicted_group'].nunique()}")
print(f"Records retained (pred_keep): {df['pred_keep'].sum()}")
print(f"Records removed: {(~df['pred_keep']).sum()}")
df[['recordid', 'predicted_group', 'pred_keep']].head(10)

Total records: 53001
Predicted clusters: 36211
Records retained (pred_keep): 36211
Records removed: 16790


,recordid,predicted_group,pred_keep
0,15153,15148,True
1,29964,26953,True
2,22076,22062,True
3,10934,10931,True
4,36237,20445,False
5,20457,20445,True
6,26394,25588,True
7,26481,25614,True
8,17905,17895,True
9,17904,17894,True


### Step 1: Build gold-standard labels

The gold standard is derived directly from the `duplicateid` column:

- Records that share a `duplicateid` belong to the same true duplicate group. Exactly one record per group (the one with the lowest `recordid`) is designated as the **true keep**; all others are **true duplicates that should be removed**.
- Records with a missing `duplicateid` are singletons — true uniques — and are always designated as **true keeps**.

In [10]:
# Records sharing a duplicateid are a true duplicate group; one per group should be kept.
# Records with a unique/missing duplicateid are true uniques and should be kept.

# Mark exactly one record per duplicateid group as the 'true keep'
df['true_cluster'] = df['duplicateid'].fillna(df['recordid'].astype(str)).astype(str)
df['true_keep'] = df.groupby('true_cluster')['recordid'].transform('min') == df['recordid']

n_true_dupes = (~df['true_keep']).sum()
n_true_unique = df['true_keep'].sum()
print(f"Gold standard: {n_true_unique} unique records to keep, {n_true_dupes} duplicates to remove")

Gold standard: 36146 unique records to keep, 16855 duplicates to remove


### Step 2: Compute the record-level confusion matrix

Having established both `true_removed` (gold standard) and `pred_removed` (algorithm output) as boolean columns on every record, the four cells of the confusion matrix follow directly from their intersection. The printed output shows absolute counts alongside sensitivity, specificity, and precision so the trade-off at the chosen threshold is immediately readable.

In [11]:
# true_removed = citation IS a duplicate and should have been removed
# pred_removed = citation WAS removed by the algorithm

df['true_removed'] = ~df['true_keep']
df['pred_removed'] = ~df['pred_keep']

TP = int(( df['true_removed'] &  df['pred_removed']).sum())  # duplicates correctly removed
FP = int((~df['true_removed'] &  df['pred_removed']).sum())  # uniques wrongly removed
TN = int((~df['true_removed'] & ~df['pred_removed']).sum())  # uniques correctly kept
FN = int(( df['true_removed'] & ~df['pred_removed']).sum())  # duplicates wrongly kept

sensitivity = TP / (TP + FN) if (TP + FN) else 0.0  # recall: fraction of dupes found
specificity = TN / (TN + FP) if (TN + FP) else 0.0  # fraction of uniques kept
precision   = TP / (TP + FP) if (TP + FP) else 0.0  # fraction removed that were real dupes

print(f"Threshold: {THRESHOLD}")
print(f"TP (duplicates correctly removed): {TP}")
print(f"FP (unique records wrongly removed): {FP}")
print(f"TN (unique records correctly kept): {TN}")
print(f"FN (duplicates missed / wrongly kept): {FN}")
print()
print(f"Sensitivity (recall): {sensitivity:.4f}  — % of true duplicates removed")
print(f"Specificity:          {specificity:.4f}  — % of true uniques kept")
print(f"Precision:            {precision:.4f}  — % of removed records that were real duplicates")

Threshold: 0.8
TP (duplicates correctly removed): 16766
FP (unique records wrongly removed): 24
TN (unique records correctly kept): 36122
FN (duplicates missed / wrongly kept): 89

Sensitivity (recall): 0.9947  — % of true duplicates removed
Specificity:          0.9993  — % of true uniques kept
Precision:            0.9986  — % of removed records that were real duplicates


### Step 3: Threshold sweep at the record level

The single-threshold evaluation above gives a snapshot, but the optimal operating point may differ when measured at the record level versus the pair level (because clustering can propagate errors across records). This sweep reruns the full cluster → keep/remove pipeline for every threshold in `THRESHOLDS` and reports the record-level confusion matrix for each.

The resulting `record_metrics_df` table is the primary output for deciding which threshold to carry forward into production or unseen-dataset testing. A threshold that looks good at the pair level may look worse here if it causes aggressive over-clustering (many FPs) or under-clustering (many FNs).

In [12]:
# --- Sweep across thresholds: record-level confusion matrix for each ---
import pandas as pd

def record_level_metrics_for_threshold(df_orig, scored_df, threshold):
    df_t = df_orig.copy()
    # Cluster
    G = nx.Graph()
    G.add_nodes_from(set(df_t['recordid']))
    pos_pairs = scored_df[scored_df['prob'] >= threshold][['id_a', 'id_b']].to_numpy()
    G.add_edges_from(pos_pairs)
    record_to_group = {}
    for gid, comp in enumerate(nx.connected_components(G)):
        for rid in comp:
            record_to_group[rid] = gid
    df_t['predicted_group'] = df_t['recordid'].map(record_to_group)
    df_t['pred_keep'] = df_t.groupby('predicted_group')['recordid'].transform('min') == df_t['recordid']
    # Gold standard
    df_t['true_cluster'] = df_t['duplicateid'].fillna(df_t['recordid'].astype(str)).astype(str)
    df_t['true_keep'] = df_t.groupby('true_cluster')['recordid'].transform('min') == df_t['recordid']
    # Confusion matrix
    true_rem = ~df_t['true_keep']
    pred_rem = ~df_t['pred_keep']
    tp = int(( true_rem &  pred_rem).sum())
    fp = int((~true_rem &  pred_rem).sum())
    tn = int((~true_rem & ~pred_rem).sum())
    fn = int(( true_rem & ~pred_rem).sum())
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    return {'threshold': threshold, 'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn,
            'sensitivity': sens, 'specificity': spec, 'precision': prec}

df_base = load_data(DATA_PATH)  # fresh copy without computed columns
record_metrics = [record_level_metrics_for_threshold(df_base, scored_df, t) for t in THRESHOLDS]
record_metrics_df = pd.DataFrame(record_metrics)
record_metrics_df

,threshold,TP,FP,TN,FN,sensitivity,specificity,precision
0,0.70,16813,65,36081,42,0.997508,0.998202,0.996149
1,0.75,16799,44,36102,56,0.996678,0.998783,0.997388
2,0.80,16766,24,36122,89,0.994720,0.999336,0.998571
3,0.85,16674,16,36130,181,0.989261,0.999557,0.999041
4,0.90,16530,9,36137,325,0.980718,0.999751,0.999456
